In [0]:
# =============================================================
# cleaner.py
# Layer     : Silver
# Purpose   : Cleaning rules + type casting only.
#             One function per table. Called by silver_loader.
#
# STRICT RULE: No spark.read. No spark.write. No Delta.
#              No schema definitions. Only column transformations.
#              If you see DeltaTable here → wrong file.
# =============================================================

from pyspark.sql.functions import (
    col, when, abs, lit,
    to_timestamp, to_date
)
from pyspark.sql.types import (
    DoubleType, IntegerType, BooleanType
)


# ── Dispatch ──────────────────────────────────────────────────────────────────

def clean(df, table_name: str):
    """
    Entry point called by silver_loader.
    Routes to the correct table-specific cleaning function.
    Returns a cleaned, properly typed DataFrame.
    """
    _dispatch = {
        "owners":               _clean_owners,
        "pets":                 _clean_pets,
        "products":             _clean_products,
        "firmware":             _clean_firmware,
        "batches":              _clean_batches,
        "devices":              _clean_devices,
        "sales":                _clean_sales,
        "device_snapshots":     _clean_device_snapshots,
        "activity_snapshots":   _clean_activity_snapshots,
        "health_snapshots":     _clean_health_snapshots,
        "feeding_events":       _clean_feeding_events,
        "hydration_events":     _clean_hydration_events,
        "device_failures":      _clean_device_failures,
        "firmware_deployments": _clean_firmware_deployments,
    }

    if table_name not in _dispatch:
        raise KeyError(
            f"[cleaner] No cleaning function for '{table_name}'. "
            f"Available: {sorted(_dispatch.keys())}"
        )

    fn     = _dispatch[table_name]
    result = fn(df)
    print(f"[cleaner] Cleaning complete for '{table_name}'.")
    return result


# ── DIMENSION TABLES ──────────────────────────────────────────────────────────
# Dimensions have no injected anomalies — cleaning is type casting only.

def _clean_owners(df):
    return (df
        .withColumn("registration_date",
            to_timestamp(col("registration_date")))
    )

def _clean_pets(df):
    return (df
        .withColumn("birth_date",
            to_date(col("birth_date")))
        .withColumn("weight_kg",
            col("weight_kg").cast(DoubleType()))  # Safe cast
        .withColumn("weight_kg",
            when(col("weight_kg") < 0, None)
            .otherwise(col("weight_kg")))
        # Standardise gender casing
        .withColumn("gender",
            when(col("gender").isin("male", "MALE", "M"),   lit("Male"))
            .when(col("gender").isin("female", "FEMALE", "F"), lit("Female"))
            .otherwise(col("gender")))
    )

def _clean_products(df):
    return (df
        .withColumn("manufacturing_cost",
            col("manufacturing_cost").cast(DoubleType()))  # Safe cast
        .withColumn("msrp",
            col("msrp").cast(DoubleType()))  # Safe cast
    )

def _clean_firmware(df):
    return (df
        .withColumn("release_date",
            to_date(col("release_date")))
    )

def _clean_batches(df):
    return (df
        .withColumn("production_date",
            to_date(col("production_date")))
    )

def _clean_devices(df):
    return (df
        .withColumn("purchase_date",
            to_date(col("purchase_date")))
        .withColumn("activation_date",
            to_date(col("activation_date")))
        .withColumn("warranty_end_date",
            to_date(col("warranty_end_date")))
        # activation_date must be >= purchase_date
        .withColumn("activation_date",
            when(col("activation_date") < col("purchase_date"),
                 col("purchase_date"))
            .otherwise(col("activation_date")))
    )


# ── FACT TABLES ───────────────────────────────────────────────────────────────
# Facts have injected anomalies — cleaning fixes them then casts types.

def _clean_sales(df):
    return (df
        .withColumn("sale_date",
            to_date(col("sale_date")))
        .withColumn("sale_price",
            col("sale_price").cast(DoubleType()))  # Safe cast
        .withColumn("discount_amount",
            col("discount_amount").cast(DoubleType()))  # Safe cast
        # Fix 1: negative sale_price → null
        .withColumn("sale_price",
            when(col("sale_price") < 0, None)
            .otherwise(col("sale_price")))
        # Fix 2: discount > sale_price → cap at sale_price
        .withColumn("discount_amount",
            when(col("discount_amount") > col("sale_price"),
                 col("sale_price"))
            .otherwise(col("discount_amount")))
        # Fix 3: null or blank sales_channel → Unknown
        .withColumn("sales_channel",
            when(
                col("sales_channel").isNull() | (col("sales_channel") == ""),
                lit("Unknown"))
            .otherwise(col("sales_channel")))
        .dropDuplicates(["sale_id"])
    )

def _clean_device_snapshots(df):
    return (df
        .withColumn("snapshot_timestamp",
            to_timestamp(col("snapshot_timestamp")))
        .withColumn("battery_pct",
            col("battery_pct").cast(DoubleType()))  # Safe cast
        .withColumn("signal_strength",
            col("signal_strength").cast(IntegerType()))  # Safe cast
        .withColumn("temperature_c",
            col("temperature_c").cast(DoubleType()))  # Safe cast
        # Fix 1: battery out of range → null
        .withColumn("battery_pct",
            when((col("battery_pct") < 0) | (col("battery_pct") > 100), None)
            .otherwise(col("battery_pct")))
        # Fix 2: temperature = 0.0 → null (sensor dropout)
        .withColumn("temperature_c",
            when(col("temperature_c") == 0.0, None)
            .otherwise(col("temperature_c")))
        # Fix 3: remove duplicate device + timestamp rows
        .dropDuplicates(["device_id", "snapshot_timestamp"])
    )

def _clean_activity_snapshots(df):
    return (df
        .withColumn("snapshot_timestamp",
            to_timestamp(col("snapshot_timestamp")))
        .withColumn("steps",
            col("steps").cast(IntegerType()))  # Safe cast
        .withColumn("distance_km",
            col("distance_km").cast(DoubleType()))  # Safe cast
        .withColumn("active_minutes",
            col("active_minutes").cast(IntegerType()))  # Safe cast
        # Fix 1: negative steps → abs
        .withColumn("steps",
            when(col("steps") < 0, abs(col("steps")))
            .otherwise(col("steps")))
        # Fix 2: active_minutes > 60 → cap at 60 (hourly grain)
        .withColumn("active_minutes",
            when(col("active_minutes") > 60, lit(60))
            .otherwise(col("active_minutes")))
        # Fix 3: remove duplicates
        .dropDuplicates(["device_id", "snapshot_timestamp"])
    )

def _clean_health_snapshots(df):
    from pyspark.sql.functions import expr, to_timestamp, col
    
    return (df
        .withColumn("snapshot_timestamp", 
            to_timestamp(col("snapshot_timestamp")))
        # Safe cast version-proof wrappers
        .withColumn("heart_rate", expr("try_cast(heart_rate AS double)"))
        .withColumn("pulse_rate", expr("try_cast(pulse_rate AS double)"))
        .withColumn("body_temperature", expr("try_cast(body_temperature AS double)"))
        # Keep this safe filter as a backup safeguard
        .dropna(subset=["heart_rate", "pulse_rate", "body_temperature"], how="all")
    )

def _clean_feeding_events(df):
    return (df
        .withColumn("event_timestamp",
            to_timestamp(col("event_timestamp")))
        .withColumn("food_dispensed_grams",
            col("food_dispensed_grams").cast(DoubleType()))  # Safe cast
        # Fix 1: food <= 0 → null (empty hopper / bad sensor)
        .withColumn("food_dispensed_grams",
            when(col("food_dispensed_grams") <= 0, None)
            .otherwise(col("food_dispensed_grams")))
        # Fix 2: food > 1000g → null (extreme over-dispense)
        .withColumn("food_dispensed_grams",
            when(col("food_dispensed_grams") > 1000, None)
            .otherwise(col("food_dispensed_grams")))
        # Fix 3: remove duplicates
        .dropDuplicates(["feed_event_id"])
    )

def _clean_hydration_events(df):
    from pyspark.sql.functions import expr, when, col, lit, to_timestamp
    
    return (df
        .withColumn("event_timestamp", 
            to_timestamp(col("event_timestamp")))
        # Safe cast version-proof wrapper
        .withColumn("water_consumed_ml", 
            expr("try_cast(water_consumed_ml AS double)"))
        # Telemetry Fix: If water consumed is negative, ground it to 0.0
        .withColumn("water_consumed_ml",
            when(col("water_consumed_ml") < 0, lit(0.0))
            .otherwise(col("water_consumed_ml")))
        # REMOVED: .dropna(subset=["water_consumed_ml"])
    )

def _clean_device_failures(df):
    return (df
        .withColumn("failure_timestamp",
            to_timestamp(col("failure_timestamp")))
        .withColumn("downtime_minutes",
            col("downtime_minutes").cast(IntegerType()))  # Safe cast
        .withColumn("severity_score",
            col("severity_score").cast(IntegerType()))  # Safe cast
        # Fix 1: severity outside 1-10 → null
        .withColumn("severity_score",
            when((col("severity_score") < 1) | (col("severity_score") > 10), None)
            .otherwise(col("severity_score")))
        # Fix 2: negative downtime → abs
        .withColumn("downtime_minutes",
            when(col("downtime_minutes") < 0, abs(col("downtime_minutes")))
            .otherwise(col("downtime_minutes")))
        # Fix 3: null or blank failure_type → Unknown
        .withColumn("failure_type",
            when(
                col("failure_type").isNull() | (col("failure_type") == ""),
                lit("Unknown"))
            .otherwise(col("failure_type")))
    )

def _clean_firmware_deployments(df):
    return (df
        .withColumn("deployment_timestamp",
            to_timestamp(col("deployment_timestamp")))
        .withColumn("deployment_duration_seconds",
            col("deployment_duration_seconds").cast(IntegerType()))  # Safe cast
        .withColumn("rollback_flag",
            col("rollback_flag").cast(BooleanType()))  # Safe cast
        # Fix 1: rollback_flag=True but status != Rolled Back → False
        .withColumn("rollback_flag",
            when(
                (col("rollback_flag") == True) &
                (col("deployment_status") != "Rolled Back"),
                lit(False))
            .otherwise(col("rollback_flag")))
        # Fix 2: negative duration → abs
        .withColumn("deployment_duration_seconds",
            when(col("deployment_duration_seconds") < 0,
                 abs(col("deployment_duration_seconds")))
            .otherwise(col("deployment_duration_seconds")))
        # Fix 3: null firmware_id → drop (orphan deployment)
        .dropna(subset=["firmware_id"])
    )

print("[cleaner] Loaded. Cleaning functions registered for 14 tables.")